# DeepCSI on real COST2100 data — Colab GPU runner

Trains the DeepCSI autoencoder at CR = 4 / 16 / 32 on the **COST2100** CSI feedback
benchmark (the dataset released with CsiNet), then evaluates against a DCT baseline
and against published results.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

The dataset download happens on Google's network inside this notebook — nothing
lands on your laptop. At the end you download ~25 MB of artifacts for the local
FastAPI + Streamlit demo.

Published reference numbers on this benchmark (indoor):

| CR | CsiNet | CRNet |
|---:|-------:|------:|
| 4  | -17.36 dB | -26.99 dB |
| 16 |  -8.65 dB | -11.35 dB |
| 32 |  -6.24 dB |  -8.93 dB |

If our numbers land in this neighbourhood, they are real. Anything near -38 dB
means a metric is being computed on offset data again.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

## 2. Get the code

**Option A** — if you pushed the branch to a remote you can reach:

In [ ]:
# Edit REPO to your fork if you pushed there.
REPO = "https://github.com/Tharwathabib/DeepCSI.git"
BRANCH = "karim/real-data"

!git clone --branch {BRANCH} {REPO} deepcsi || git clone {REPO} deepcsi
%cd deepcsi
!git log --oneline -3

**Option B** — no push access. Zip the worktree locally and upload it here:

```bash
# on your laptop, from Downloads/DeepCSI/
tar --exclude=.git --exclude=data --exclude=results --exclude=models/weights \
    -czf deepcsi-karim.tgz DeepCSI-karim
```

Then run the cell below instead of Option A.

In [ ]:
# Option B only -- skip if Option A worked.
from google.colab import files
up = files.upload()            # pick deepcsi-karim.tgz
!tar -xzf deepcsi-karim.tgz
%cd DeepCSI-karim
!ls

## 3. Install dependencies

In [ ]:
!pip install -q gdown h5py
# torch, numpy, scipy, pandas, matplotlib are preinstalled on Colab.
import scipy, pandas, h5py
print("ok")

## 4. Download COST2100

The canonical copies are a Google Drive folder (from the CRNet repo) and a Dropbox
folder (from the CsiNet repo). Drive can throw a quota error on heavily-downloaded
files; if both fail, mirror the four indoor files into your own Drive once and
`gdown` that folder instead.

We only need the four **indoor** files:
`DATA_Htrainin.mat`, `DATA_Hvalin.mat`, `DATA_Htestin.mat` (and `DATA_HtestFin_all.mat`,
which we do not use here).

In [ ]:
import os, glob
os.makedirs("data/raw/COST2100", exist_ok=True)

GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1_lAMLk_5k1Z8zJQlTr5NRnSD6ACaNRtj"
!gdown --folder {GDRIVE_FOLDER} -O data/raw/COST2100 --remaining-ok || echo "gdown folder failed"

found = glob.glob("data/raw/COST2100/**/*.mat", recursive=True)
print(f"\n{len(found)} .mat files:")
for f in sorted(found):
    print(f"  {f}  ({os.path.getsize(f)/1e6:.0f} MB)")

In [ ]:
# Flatten any nested folder gdown created, so the .mat files sit directly in data/raw/COST2100.
import shutil, glob, os
for f in glob.glob("data/raw/COST2100/**/*.mat", recursive=True):
    dest = os.path.join("data/raw/COST2100", os.path.basename(f))
    if os.path.abspath(f) != os.path.abspath(dest):
        shutil.move(f, dest)
print(sorted(os.path.basename(p) for p in glob.glob("data/raw/COST2100/*.mat")))

In [ ]:
# FALLBACK -- only run if the Drive download failed.
# Dropbox serves the whole folder as a zip; this is large, so prefer Drive.
!wget -q --show-progress -O cost2100.zip \
  "https://www.dropbox.com/scl/fo/tqhriijik2p76j7kfp9jl/h?rlkey=4r1zvjpv4lh5h4fpt7lbpus8c&dl=1"
!unzip -o -j cost2100.zip "*.mat" -d data/raw/COST2100
!ls -la data/raw/COST2100

## 5. Prepare the tensors

Converts `(N, 2048)` MATLAB arrays into the `(N, 2, 32, 32)` float32 tensors the
pipeline consumes, and writes `norm_params.json` recording the CsiNet offset
convention (0.5 is the complex zero point). The train/val/test split ships with
the dataset, so there is no re-splitting and no normalisation leakage.

`--subsample 20000` keeps the run fast. Drop the flag to use all 100,000.

In [ ]:
!python data/prepare_cost2100.py \
    --mat-dir data/raw/COST2100 \
    --output-dir data/processed_cost2100 \
    --environment indoor \
    --subsample 20000

## 6. Train

CR=4 first: it is the easiest configuration and acts as the gate. If CR=4 does not
get near the CsiNet reference (-17 dB), stop and debug rather than training the
other two.

In [ ]:
COMMON = "--data-dir data/processed_cost2100 --batch-size 200 --epochs 40 --patience 10 --amp"
!python models/train.py --compression-ratio 4 {COMMON}

In [ ]:
!python models/train.py --compression-ratio 16 {COMMON}

In [ ]:
!python models/train.py --compression-ratio 32 {COMMON}

## 7. Evaluate

Produces `results/metrics.csv`, the comparison pivots and the figures. The NMSE
table is printed in the log-of-mean convention on the de-offset channel, which is
what CsiNet and CRNet report — so the columns are directly comparable.

In [ ]:
!python models/evaluate.py \
    --data-dir data/processed_cost2100 \
    --weights-dir models/weights \
    --results-dir results

In [ ]:
import pandas as pd
df = pd.read_csv("results/metrics.csv")
display(df)

from IPython.display import Image, display as d
for fig in ["nmse_vs_cr.png", "beamforming_gain_vs_cr.png", "reconstruction_heatmaps.png"]:
    try:
        d(Image(f"results/figures/{fig}"))
    except Exception as e:
        print(fig, "->", e)

## 8. Sanity check before you trust these numbers

Three things to confirm. If any fails, the result is not presentable.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("results/metrics.csv")
deep = df[df.method == "DeepCSI"].sort_values("compression_ratio")
col = "nmse_db_aggregate" if "nmse_db_aggregate" in deep else "nmse_db_mean"

ok = True

# 1. NMSE must get worse as compression increases. A flat curve means the
#    bottleneck is not binding and something is wrong.
nmse = deep[col].tolist()
mono = all(a < b for a, b in zip(nmse, nmse[1:]))
print(f"1. NMSE degrades with CR : {'PASS' if mono else 'FAIL'}  {nmse}")
ok &= mono

# 2. Nothing near -38 dB: that was the signature of measuring NMSE on
#    DC-offset data, which inflates the denominator by ~35 dB.
sane = all(n > -30 for n in nmse)
print(f"2. No implausible -38 dB : {'PASS' if sane else 'FAIL'}")
ok &= sane

# 3. rho must vary across CR. Pinned at ~99.98% means the de-offset was skipped.
gains = deep["beamforming_gain_mean"].tolist()
varies = (max(gains) - min(gains)) > 1e-3
print(f"3. rho varies across CR  : {'PASS' if varies else 'FAIL'}  {[round(g,4) for g in gains]}")
ok &= varies

print("\n" + ("ALL CHECKS PASSED" if ok else "SOMETHING IS WRONG -- do not present these numbers"))

## 9. Download the artifacts

Everything the local demo needs. The test set is trimmed to 2,000 samples to keep
the download small — that is what the dashboard's sample selector uses.

In [ ]:
import numpy as np, shutil, os, json

os.makedirs("export/data", exist_ok=True)
os.makedirs("export/models/weights", exist_ok=True)

test = np.load("data/processed_cost2100/test.npy")
np.save("export/data/test.npy", test[:2000])
shutil.copy("data/processed_cost2100/norm_params.json", "export/data/norm_params.json")

# norm_params records the full split sizes; correct num_test for the trimmed copy.
p = json.load(open("export/data/norm_params.json"))
p["num_test"] = int(min(2000, len(test)))
p["note"] = "test.npy trimmed to the first 2000 samples for the local demo"
json.dump(p, open("export/data/norm_params.json", "w"), indent=2)

for cr in (4, 16, 32):
    src = f"models/weights/deepcsi_cr{cr}.pt"
    if os.path.exists(src):
        shutil.copy(src, f"export/models/weights/deepcsi_cr{cr}.pt")
shutil.copytree("results", "export/results", dirs_exist_ok=True)

!cd export && zip -qr ../deepcsi_artifacts.zip . && cd ..
print(f"deepcsi_artifacts.zip -> {os.path.getsize('deepcsi_artifacts.zip')/1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download("deepcsi_artifacts.zip")

## 10. Back on your laptop

```bash
cd Downloads/DeepCSI/DeepCSI-karim
unzip -o ~/Downloads/deepcsi_artifacts.zip -d .
mv data/test.npy data/norm_params.json data/processed_cost2100/   # if needed

set DATA_DIR=data/processed_cost2100        # PowerShell: $env:DATA_DIR="data/processed_cost2100"
python preflight.py
uvicorn backend.app:app --reload --port 8000
streamlit run frontend/app.py
```